In [ ]:
# 01_preprocessing_grouping_model.ipynb
# Load the synthetic card-usage data, build the 12-feature governance partition,
# define the intersectional groups with data-driven tenure cutoffs, train the
# Random Forest (and three additional classifiers for robustness), and cache all
# artifacts needed by the downstream notebooks. Paths are relative; nothing that
# could identify the authors or the local machine is printed.

import os, glob, zipfile, pickle
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score, roc_auc_score

# ---- Relative project paths (run from repo root or from notebooks/) ----
ROOT = Path.cwd(); ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
DATA_DIR = ROOT / "data"
CACHE = ROOT / "results" / "cache"; CACHE.mkdir(parents=True, exist_ok=True)
ENC = "utf-8-sig"
RANDOM_STATE = 42

# ---- Locate the three required source tables by their (Korean) header signature.
#      External credit-bureau tables are intentionally excluded (limited-information setting).
csv_paths = sorted(glob.glob(str(DATA_DIR / "**" / "*.csv"), recursive=True))
SIGNATURES = {"member": "VIP등급코드", "approval": "이용금액_신용_R3M", "credit": "카드이용한도금액"}
def classify(path):
    cols = pd.read_csv(path, nrows=0, encoding=ENC).columns
    for label, sig in SIGNATURES.items():
        if sig in cols: return label
    return "unknown"
file_index = {}
for p in csv_paths:
    file_index.setdefault(classify(p), []).append(p)

MEMBER_COLS = {"기준년월":"job_mon","발급회원번호":"member_no","남녀구분코드":"code_gender",
    "연령":"age","VIP등급코드":"code_vip","거주시도명":"add_home_si",
    "입회경과개월수_신용":"mn_credit_tenure","Life_Stage":"code_life","회원여부_연체":"yn_delinquent"}
APPROVAL_COLS = {"발급회원번호":"member_no","이용금액_신용_R3M":"amt_credit_r3m","이용금액_업종기준":"amt_usage_r3m",
    "이용가맹점수":"cnt_merchant_r3m","이용금액_쇼핑":"amt_shop","이용금액_요식":"amt_eat","이용금액_여유생활":"amt_leisure"}
CREDIT_COLS = {"발급회원번호":"member_no","카드이용한도금액":"amt_credit_limit"}
def load_latest(label, colmap):
    latest = max(file_index[label], key=lambda p: os.path.basename(p)[:6])
    return pd.read_csv(latest, encoding=ENC, usecols=list(colmap.keys())).rename(columns=colmap)

member = load_latest("member", MEMBER_COLS)
approval = load_latest("approval", APPROVAL_COLS)
credit = load_latest("credit", CREDIT_COLS)
df = (member.merge(approval.drop(columns=["job_mon"], errors="ignore"), on="member_no", how="inner")
            .merge(credit.drop(columns=["job_mon"], errors="ignore"), on="member_no", how="inner"))

# ---- Binarize the premium (VIP) target ----
df["code_vip"] = df["code_vip"].astype(str).str.strip()
df["VIP"] = (~df["code_vip"].isin(["_", "nan", ""])).astype(int)

# ---- Intersectional groups with data-driven tenure cutoffs ----
TENURE_SHORT = df["mn_credit_tenure"].quantile(0.25)
TENURE_LONG = df["mn_credit_tenure"].quantile(0.75)
MALE, FEMALE = 1, 2
YOUNG_BANDS, OLD_BANDS = ["30대"], ["60대", "70대이상"]
def assign_group(row):
    long_h = row["mn_credit_tenure"] >= TENURE_LONG
    short_h = row["mn_credit_tenure"] <= TENURE_SHORT
    if row["age"] in YOUNG_BANDS and row["code_gender"] == MALE and long_h: return "advantaged"
    if row["age"] in OLD_BANDS and row["code_gender"] == FEMALE and short_h: return "disadvantaged"
    return "other"
df["group"] = df.apply(assign_group, axis=1)

# ---- Feature governance: 12 features = 5 immutable + 6 actionable + 1 non-controllable ----
IMMUTABLE = ["code_gender", "age", "add_home_si", "mn_credit_tenure", "code_life"]
ACTIONABLE = ["amt_credit_r3m", "amt_usage_r3m", "cnt_merchant_r3m", "amt_shop", "amt_eat", "amt_leisure"]
NON_CONTROLLABLE = ["amt_credit_limit"]
FEATURES = IMMUTABLE + ACTIONABLE + NON_CONTROLLABLE
TARGET = "VIP"

model_df = df[FEATURES + [TARGET, "group"]].copy()
AGE_ORDER = ["20대", "30대", "40대", "50대", "60대", "70대이상"]
model_df["age"] = pd.Categorical(model_df["age"], categories=AGE_ORDER, ordered=True).codes
for c in ["code_gender", "add_home_si", "code_life"]:
    model_df[c] = model_df[c].astype(str).astype("category").cat.codes

# ---- Stratified large sample (preserve VIP ratio and group mix) ----
SAMPLE_N = 200000
strat_key = model_df["group"].astype(str) + "_" + model_df[TARGET].astype(str)
sample_df, _ = train_test_split(model_df, train_size=SAMPLE_N, stratify=strat_key, random_state=RANDOM_STATE)
X = sample_df[FEATURES]; y = sample_df[TARGET]
strat2 = sample_df["group"].astype(str) + "_" + y.astype(str)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=strat2, random_state=RANDOM_STATE)
group_test = sample_df.loc[X_test.index, "group"]

# ---- Train the primary classifier (Random Forest) ----
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
proba = rf.predict_proba(X_test[FEATURES])[:, 1]
metrics = {"F1": f1_score(y_test, (proba >= 0.5).astype(int)),
           "recall": recall_score(y_test, (proba >= 0.5).astype(int)),
           "AUC": roc_auc_score(y_test, proba)}
cdr_tbl = (pd.DataFrame({"group": group_test.values, "proba": proba})
           .groupby("group")["proba"].mean().reindex(["advantaged", "disadvantaged", "other"]) * 100).round(3)

# ---- Model-metrics table to results/tables ----
TAB = ROOT / "results" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{"metric": k, "value": round(v, 4)} for k, v in metrics.items()]).to_csv(TAB / "table0_model_metrics.csv", index=False)

# ---- Checkpoint everything the downstream notebooks need ----
artifacts = {"rf": rf, "X_train": X_train, "y_train": y_train, "X_test": X_test, "y_test": y_test,
    "group_test": group_test, "FEATURES": FEATURES, "IMMUTABLE": IMMUTABLE, "ACTIONABLE": ACTIONABLE,
    "NON_CONTROLLABLE": NON_CONTROLLABLE, "TARGET": TARGET, "cdr_table": cdr_tbl,
    "cutoffs": {"short": float(TENURE_SHORT), "long": float(TENURE_LONG)}}
with open(CACHE / "nb1_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("Notebook 01 complete. Model F1/recall/AUC and group CDR table computed; artifacts cached.")
print(pd.DataFrame([metrics]).round(4).to_string(index=False))
